In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
DRIVE_DIR = '/content/drive/MyDrive/mood2music'
print(os.listdir(DRIVE_DIR))

['catalog.parquet', 'track_index.faiss', 'track_id_map.pkl', 'fma_needed.zip', 'clap_checkpoint.pkl', 'track_id_map_clap.pkl', 'track_index_clap.faiss']


In [5]:
!pip install sentence-transformers faiss-cpu librosa -q

import numpy as np
import faiss
import pickle
import duckdb
from sentence_transformers import SentenceTransformer
import torch
from transformers import ClapModel, ClapProcessor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 108.8 MB/s eta 0:00:00


In [6]:
CATALOG_PATH = f'{DRIVE_DIR}/catalog.parquet'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cuda


In [7]:
catalog = duckdb.query(f"""
    SELECT track_id, title, artist_name, genre_top,
           valence, energy, valence_reliable
    FROM '{CATALOG_PATH}'
    WHERE valence_reliable = true
""").df()

print(f"Catalog loaded: {len(catalog)} tracks")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Catalog loaded: 11868 tracks


In [8]:
model_2a = SentenceTransformer('all-MiniLM-L6-v2')
index_2a = faiss.read_index(f'{DRIVE_DIR}/track_index.faiss')
with open(f'{DRIVE_DIR}/track_id_map.pkl', 'rb') as f:
    map_2a = pickle.load(f)

print(f"Phase 2A loaded — index size: {index_2a.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Phase 2A loaded — index size: 11868


In [9]:
model_clap = ClapModel.from_pretrained('laion/clap-htsat-fused').to(device)
processor_clap = ClapProcessor.from_pretrained('laion/clap-htsat-fused')
model_clap.eval()

index_clap = faiss.read_index(f'{DRIVE_DIR}/track_index_clap.faiss')
with open(f'{DRIVE_DIR}/track_id_map_clap.pkl', 'rb') as f:
    map_clap = pickle.load(f)

print(f"CLAP loaded — index size: {index_clap.ntotal}")

config.json:   0%|          | 0.00/5.42k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  614MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/477 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

CLAP loaded — index size: 4816


In [10]:
def track_metadata(track_id, catalog):
    row = catalog[catalog['track_id'] == track_id].iloc[0]
    return {
        'track_id': int(row['track_id']),
        'title':    row['title'],
        'artist':   row['artist_name'],
        'genre':    row['genre_top'],
        'valence':  round(float(row['valence']), 3),
        'energy':   round(float(row['energy']), 3),
    }


In [17]:
def embed_text_clap(query_text):
    with torch.no_grad():
        inputs = processor_clap(
            text=[query_text], return_tensors='pt', padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        embed = model_clap.get_text_features(**inputs)
        if hasattr(embed, 'pooler_output'):
            embed = embed.pooler_output
    return embed.cpu().numpy()

In [12]:
def search_2a(query_text, k=5):
    query_vec = model_2a.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, positions = index_2a.search(query_vec, k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        meta = track_metadata(map_2a[pos], catalog)
        meta['similarity'] = round(float(score), 3)
        results.append(meta)
    return results

In [13]:
def search_clap(query_text, k=5):
    query_vec = embed_text_clap(query_text)
    faiss.normalize_L2(query_vec)
    scores, positions = index_clap.search(query_vec, k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        meta = track_metadata(map_clap[pos], catalog)
        meta['similarity'] = round(float(score), 3)
        results.append(meta)
    return results

In [21]:
def search_hybrid(query_text, k=5, candidate_pool=50,
                   w_clap=0.7, w_valence=0.3, w_energy=0.2,
                   n_valence_ref=5):
    """
    Two-stage retrieval with dual-axis reranking.

    Scoring:
        final_score = w_clap   × clap_similarity
                    - w_valence × |track_valence - target_valence|
                    - w_energy  × |track_energy  - target_energy|

    target_valence and target_energy both derived from Phase 2A top-n
    results — data-driven, no hand-coded mood mapping.

    w_clap=0.7, w_valence=0.3, w_energy=0.2:
        CLAP dominates (broad semantic match).
        Valence penalty is stronger than energy penalty because
        CLAP's valence weakness is more severe than its energy weakness.
        Energy penalty is lighter — CLAP already does reasonably well
        on energy, so we correct gently rather than override aggressively.
    """
    # Step 1 — derive targets from Phase 2A
    ref = search_2a(query_text, k=n_valence_ref)
    target_valence = np.mean([r['valence'] for r in ref])
    target_energy  = np.mean([r['energy']  for r in ref])

    # Step 2 — CLAP candidate pool
    query_vec = embed_text_clap(query_text)
    faiss.normalize_L2(query_vec)
    scores, positions = index_clap.search(query_vec, candidate_pool)

    candidates = []
    for score, pos in zip(scores[0], positions[0]):
        meta = track_metadata(map_clap[pos], catalog)
        meta['clap_similarity'] = round(float(score), 3)
        candidates.append(meta)

    # Step 3 — rerank with dual penalty
    for c in candidates:
        v_gap = abs(c['valence'] - target_valence)
        e_gap = abs(c['energy']  - target_energy)
        c['valence_gap']    = round(v_gap, 3)
        c['energy_gap']     = round(e_gap, 3)
        c['target_valence'] = round(target_valence, 3)
        c['target_energy']  = round(target_energy, 3)
        c['final_score']    = round(
            w_clap   * c['clap_similarity']
            - w_valence * v_gap
            - w_energy  * e_gap, 4
        )

    candidates.sort(key=lambda x: x['final_score'], reverse=True)
    return candidates[:k]

In [22]:
test_queries = [
    "I feel melancholy and tired, want something that matches my mood",
    "I need high energy music to get through a workout",
    "Something calm and peaceful for late night reading",
    "I'm feeling nostalgic and bittersweet"
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}")

    r2a  = search_2a(query, k=3)
    rclap = search_clap(query, k=3)
    rhyb  = search_hybrid(query, k=3)

    target_val = rhyb[0]['target_valence']
    target_e   = rhyb[0]['target_energy']
    print(f"\nTarget valence (from 2A): {target_val:.3f}  "
          f"Target energy (from 2A): {target_e:.3f}")
    print(f"\n{'#':<4} {'System':<8} {'Title':<35} {'val':>5} {'nrg':>5} {'score':>7}")
    print("-" * 70)

    for i in range(3):
        a = r2a[i]
        c = rclap[i]
        h = rhyb[i]

        print(f"{i+1:<4} {'2A':<8} {a['title'][:34]:<35} "
              f"{a['valence']:>5.2f} {a['energy']:>5.2f} "
              f"{a['similarity']:>7.3f}")
        print(f"{'':4} {'CLAP':<8} {c['title'][:34]:<35} "
              f"{c['valence']:>5.2f} {c['energy']:>5.2f} "
              f"{c['similarity']:>7.3f}")
        print(f"{'':4} {'Hybrid':<8} {h['title'][:34]:<35} "
              f"{h['valence']:>5.2f} {h['energy']:>5.2f} "
              f"{h['final_score']:>7.4f}")
        print()


Query: 'I feel melancholy and tired, want something that matches my mood'

Target valence (from 2A): 0.124  Target energy (from 2A): 0.553

#    System   Title                                 val   nrg   score
----------------------------------------------------------------------
1    2A       Smoke In The Blues Bar               0.09  0.21   0.593
     CLAP     A Life In A Day                      0.05  0.73   0.395
     Hybrid   A Life In A Day                      0.05  0.73  0.2198

2    2A       Guitarists Against Landmines         0.16  0.66   0.587
     CLAP     Оля Зимой                            0.78  0.38   0.358
     Hybrid   asynchrony of life                   0.07  0.52  0.2069

3    2A       Claw Back The Unamortised Discount   0.07  0.71   0.587
     CLAP     Don't Think Twice (reprise)          0.16  0.01   0.352
     Hybrid   The Dream of Sergei Prokofiev and    0.08  0.53  0.2011


Query: 'I need high energy music to get through a workout'

Target valence (from 2A)

In [4]:
import modal

with modal.enable_output():
    f = modal.Function.from_name("mood2music-clap", "encode_text")
    result = f.remote("I feel melancholy and tired")

print(f"Vector length: {len(result)}")
print(f"First 5 values: {result[:5]}")
print(f"Norm: {sum(x**2 for x in result) ** 0.5:.4f}")

Vector length: 512
First 5 values: [-0.03375844657421112, -0.08906194567680359, -0.09271950274705887, 0.030891304835677147, -0.010234507732093334]
Norm: 1.0000
